# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TEJAS7356/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Action queue approach

The action queue translates content performance signals into practical recommendations.

Each content record will receive:
- A priority ranking to help the team decide what to review first.
- A reason code explaining why the content was selected.
- A recommended action written in simple, actionable language.

The recommendations are intended to support human review, not automatically make changes to content.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create required folders
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

print("Libraries imported successfully.")
print("Output and figure folders are ready.")

Libraries imported successfully.
Output and figure folders are ready.


In [28]:
reason_codes = {
    "CONTENT_DECAY": "Review content freshness and relevance",
    "LOW_PERFORMANCE": "Investigate impressions, clicks and search position",
    "REFRESH_OPPORTUNITY": "Check outdated facts, links and examples",
    "PERFORMANCE_REVIEW": "Conduct a manual performance assessment"
}

reason_df = pd.DataFrame(
    [
        {
            "reason_code": code,
            "recommended_action": action
        }
        for code, action in reason_codes.items()
    ]
)

display(reason_df)

,reason_code,recommended_action
0,CONTENT_DECAY,Review content freshness and relevance
1,LOW_PERFORMANCE,"Investigate impressions, clicks and search pos..."
2,REFRESH_OPPORTUNITY,"Check outdated facts, links and examples"
3,PERFORMANCE_REVIEW,Conduct a manual performance assessment


In [29]:
print("Available DataFrames:")
for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}: {obj.shape}")
        print(obj.columns.tolist())

Available DataFrames:

reason_df: (4, 2)
['reason_code', 'recommended_action']

content_df: (30000, 47)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'reason_code', 'recommended_action', 'priority']

action_queue: (30000, 10)
['rank', 'content_id', 'client_id', 'reason_code'

In [30]:
DATA_URL = "https://raw.githubusercontent.com/TEJAS7356/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

content_df = pd.read_csv(DATA_URL)

print("Dataset shape:", content_df.shape)
print("Available columns:")
print(content_df.columns.tolist())

display(content_df.head())

Dataset shape: (30000, 44)
Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [31]:
# Inspect the signals used to generate the action queue

for col in [
    "trend_direction",
    "freshness_tier",
    "position_tier",
    "impression_tier"
]:
    print(f"\n{col}:")
    display(content_df[col].value_counts(dropna=False))

print("\nTrend percentage summary:")
display(content_df["trend_pct"].describe())

print("\nMissing values in key fields:")
display(
    content_df[
        ["trend_direction", "trend_pct", "freshness_tier",
         "position_tier", "impression_tier"]
    ].isna().sum()
)


trend_direction:


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



freshness_tier:


,count
freshness_tier,
0-30,20480
91-180,9171
31-90,175
181+,174



position_tier:


,count
position_tier,
page_1,11814
striking,7304
page_3_5,7242
top_3,2321
deep,1319


... (2 outputs left out)


Trend percentage summary:


,trend_pct
count,26612.000000
mean,-4.785969
std,473.861780
min,-100.000000
25%,-62.600000
50%,-33.500000
75%,0.000000
max,44900.000000



Missing values in key fields:


,0
trend_direction,0
trend_pct,3388
freshness_tier,0
position_tier,0
impression_tier,0


In [32]:
# Generate the rule-based content action queue

# Define the conditions for each reason code
conditions = [
    # Declining content that has not been updated recently
    (
        (content_df["trend_direction"] == "down") &
        (content_df["freshness_tier"].isin(["91-180", "181+"]))
    ),

    # Content ranking in striking distance
    (
        content_df["position_tier"] == "striking"
    ),

    # Content with low impressions and weak search positions
    (
        (content_df["impression_tier"] == "low") &
        (content_df["position_tier"].isin(["page_3_5", "deep"]))
    )
]

reason_labels = [
    "CONTENT_DECAY",
    "REFRESH_OPPORTUNITY",
    "LOW_PERFORMANCE"
]

# Assign the first matching reason code
content_df["reason_code"] = np.select(
    conditions,
    reason_labels,
    default="PERFORMANCE_REVIEW"
)

# Attach recommended actions using the existing reason_df
action_mapping = reason_df.set_index(
    "reason_code"
)["recommended_action"].to_dict()

content_df["recommended_action"] = (
    content_df["reason_code"].map(action_mapping)
)

# Define priority order for human review
priority_mapping = {
    "CONTENT_DECAY": 1,
    "REFRESH_OPPORTUNITY": 2,
    "LOW_PERFORMANCE": 3,
    "PERFORMANCE_REVIEW": 4
}

content_df["priority"] = content_df["reason_code"].map(
    priority_mapping
)

# Create the ranked action queue
action_queue = content_df[
    [
        "content_id",
        "client_id",
        "reason_code",
        "recommended_action",
        "priority",
        "trend_direction",
        "freshness_tier",
        "position_tier",
        "impression_tier"
    ]
].copy()

action_queue = action_queue.sort_values(
    by=["priority", "content_id"]
).reset_index(drop=True)

action_queue.insert(
    0,
    "rank",
    range(1, len(action_queue) + 1)
)

print("Ranked action queue generated.")
print("Total records:", len(action_queue))

display(action_queue.head(15))

print("\nReason-code distribution:")
display(action_queue["reason_code"].value_counts().to_frame("records"))

print("\nMissing recommended actions:",
      action_queue["recommended_action"].isna().sum())

Ranked action queue generated.
Total records: 30000


,rank,content_id,client_id,reason_code,recommended_action,priority,trend_direction,freshness_tier,position_tier,impression_tier
0,1,content_0003a66b9fa5,client_19581e27de,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,page_1,moderate
1,2,content_001ae9124afc,client_6208ef0f77,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,page_3_5,excellent
2,3,content_001b45a25ada,client_19581e27de,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,page_1,moderate
3,4,content_00202ac57009,client_6208ef0f77,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,striking,excellent
4,5,content_00301091b634,client_4ec9599fc2,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,page_1,low
5,6,content_0039f35523c0,client_3fdba35f04,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,page_1,moderate
6,7,content_0041426350c1,client_6208ef0f77,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,striking,low
7,8,content_0043afd4da06,client_19581e27de,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,page_1,moderate
8,9,content_00474279dd1c,client_6208ef0f77,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,page_3_5,good
9,10,content_0047824f4e13,client_19581e27de,CONTENT_DECAY,Review content freshness and relevance,1,down,91-180,striking,moderate



Reason-code distribution:


,records
reason_code,
PERFORMANCE_REVIEW,15878
REFRESH_OPPORTUNITY,5793
CONTENT_DECAY,5686
LOW_PERFORMANCE,2643



Missing recommended actions: 0


In [33]:
# Final validation of the ranked action queue

print("Total records:", len(action_queue))
print("Unique content IDs:", action_queue["content_id"].nunique())
print("Missing reason codes:", action_queue["reason_code"].isna().sum())
print("Missing actions:", action_queue["recommended_action"].isna().sum())
print("Missing priorities:", action_queue["priority"].isna().sum())
print("Duplicate ranks:", action_queue["rank"].duplicated().sum())

assert len(action_queue) == 30000
assert action_queue["content_id"].is_unique
assert action_queue["recommended_action"].notna().all()
assert action_queue["priority"].notna().all()

print("\nSection 1 validation passed!")

Total records: 30000
Unique content IDs: 30000
Missing reason codes: 0
Missing actions: 0
Missing priorities: 0
Duplicate ranks: 0

Section 1 validation passed!


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use

The content action playbook is designed to support SEO analysts, content strategists and website managers in identifying content that may require attention.

It can be used to:

- Identify content with potential performance decline.
- Prioritize pages for manual review.
- Suggest content freshness and improvement opportunities.
- Support data-driven content maintenance decisions.

### Limitations

- Model recommendations are based on historical performance data and may not reflect future results.
- Changes in impressions, clicks and search position can be influenced by external factors.
- A decline in clicks does not necessarily mean that content quality has decreased.
- The model identifies patterns and does not establish causal relationships.
- Recommendations should not be treated as guaranteed improvements.

The playbook is intended as a decision-support tool. Final content decisions must be made by human reviewers after considering business goals, search intent, content quality and relevance.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review requirements

Before acting on any recommendation, a content team member should:

- Review the page's search performance and recent traffic trends.
- Check whether the content is accurate, relevant and up to date.
- Verify search intent and whether the page satisfies user needs.
- Review the reason code and confirm that the recommended action makes sense.
- Consider business goals and discuss significant changes with the content owner.

### No-go list: Actions that should never be automated

The system must never automatically:

- Delete or publish content.
- Rewrite or modify live pages without human approval.
- Change titles, URLs or metadata without review.
- Make claims about content quality based only on performance metrics.
- Make final decisions about business priorities or content strategy.

### Human accountability

The playbook is a decision-support tool. A qualified human must review recommendations, approve changes and take responsibility for the final decision.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retraining triggers

The action playbook should be reviewed regularly to ensure its recommendations remain relevant.

The following conditions should trigger an investigation:

- **Performance drift:** A sustained change in impressions, clicks or search positions.
- **Trend changes:** Previously declining content begins improving, or stable content starts declining.
- **Freshness changes:** Content becomes outdated or remains unchanged for a long period.
- **Recommendation quality:** Human reviewers frequently reject or override the suggested actions.
- **Data quality:** Missing values, unexpected categories or changes in incoming data affect the rules.

### Review frequency

Performance signals should be monitored monthly. The rules should be reviewed when meaningful changes occur in search behaviour, data quality or recommendation outcomes.

### Retraining decision

This playbook currently uses rule-based recommendations, not a trained prediction model. Therefore, retraining is not currently applicable. The rules should instead be reviewed, validated and updated when necessary.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Monitoring checks for the action playbook

monitoring_checks = {
    "Missing trend values": content_df["trend_pct"].isna().sum(),
    "Downward trend records": (content_df["trend_direction"] == "down").sum(),
    "Older content records": content_df["freshness_tier"].isin(
        ["91-180", "181+"]
    ).sum(),
    "Missing position values": content_df["position_tier"].isna().sum(),
    "Missing impression tiers": content_df["impression_tier"].isna().sum()
}

monitoring_df = pd.DataFrame(
    list(monitoring_checks.items()),
    columns=["Monitoring check", "Records"]
)

display(monitoring_df)

print("Monitoring baseline recorded.")
print("These values can be compared with future monitoring runs.")

,Monitoring check,Records
0,Missing trend values,3388
1,Downward trend records,16262
2,Older content records,9345
3,Missing position values,0
4,Missing impression tiers,0


Monitoring baseline recorded.
These values can be compared with future monitoring runs.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exported outputs

The action playbook is exported as a CSV file so content teams can review, filter and share the ranked recommendations.

The export includes content identifiers, client identifiers, priority rankings, reason codes and recommended actions.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Export the ranked action queue

output_path = "work/outputs/content_action_playbook.csv"

action_queue.to_csv(output_path, index=False)

print("Action queue exported successfully.")
print("File:", output_path)
print("Total records exported:", len(action_queue))
print("Columns:", action_queue.columns.tolist())

Action queue exported successfully.
File: work/outputs/content_action_playbook.csv
Total records exported: 30000
Columns: ['rank', 'content_id', 'client_id', 'reason_code', 'recommended_action', 'priority', 'trend_direction', 'freshness_tier', 'position_tier', 'impression_tier']


In [38]:
# Verify the exported CSV

exported_df = pd.read_csv(
    "work/outputs/content_action_playbook.csv"
)

print("Export verification:")
print("Rows:", len(exported_df))
print("Columns:", len(exported_df.columns))
print("Unique content IDs:", exported_df["content_id"].nunique())
print("Missing reason codes:", exported_df["reason_code"].isna().sum())
print("Missing recommended actions:", exported_df["recommended_action"].isna().sum())

assert len(exported_df) == 30000
assert exported_df["content_id"].is_unique
assert exported_df["recommended_action"].notna().all()

print("\nML-10 export verification passed!")

Export verification:
Rows: 30000
Columns: 10
Unique content IDs: 30000
Missing reason codes: 0
Missing recommended actions: 0

ML-10 export verification passed!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.